# JJY 보행영상 라벨링 — 움직임 · 말하기 (v5_29 기준)

`Video Dataset/` 안의 mp4(train 1~20, test_1~3)마다 라벨을 만들어 **`label/` 폴더에 같은 경로·이름으로** 저장합니다.
예: `Video Dataset/train/1.mp4` → `label/train/1/labels_frames.jsonl`

- **motion**: `approaching` 다가옴 · `receding` 멀어짐 · `static` 가만히 있음 · `crossing` 옆으로 지나감
- **speak**: `must` 말해야 됨 · `may` 말해도 됨 · `must_not` 말하면 안 됨
- 프레임마다 트랙(tid)별로 한 줄: `{f, t, surface, road, walk_mps, objs:[{tid, cls, box, motion, speak, level, dist_m, X}]}`
- `events_runtime.json` = 실제로 말하는 문장 단위 이벤트 (v5_29와 같음)

**실행**: 런타임 → 런타임 유형 변경 → **T4 GPU** → 런타임 → **모두 실행**.
window745800 계정으로 드라이브 연결. 끊기면 다시 "모두 실행" — 끝난 영상(`label/.../done.json`)은 건너뜀.
코드·가중치: `JJY/label_code/` (pass1_* + pass2_label.py = pass2_trigger_v5_29 + 라벨 출력)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive')
VID = ROOT / 'Video Dataset'
LAB = ROOT / 'label'
CODE_SRC = ROOT / 'JJY' / 'label_code'
assert VID.exists(), 'window745800 계정 드라이브인지 확인 (Video Dataset 폴더가 없음)'
LAB.mkdir(exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install ultralytics
!rm -rf /content/code && cp -r "/content/drive/MyDrive/JJY/label_code" /content/code && ls /content/code

In [ ]:
import json, os, shutil, subprocess, time
# 영상 목록: Video Dataset 아래 mp4 전부 (AI Hub 159는 1초 간격 png라 제외)
videos = sorted(p for p in VID.rglob('*.mp4') if 'AIHub' not in str(p.relative_to(VID)) and p.is_file())
# 카메라 높이: test_1~3은 190 cm 촬영자 → 1.77 m, 나머지는 기본 1.70 m
CAM_H = {'test': '1.77'}
print(len(videos), 'videos'); [print(' ', v.relative_to(VID)) for v in videos]

In [ ]:
C = '/content/code'
W_DET, W_CLS = f'{C}/final_10class_best.pt', f'{C}/signal_state_cls_best.pt'

def run(args, env=None):
    t0 = time.time()
    r = subprocess.run(['python', '-u', *args], cwd=C, env={**os.environ, **(env or {})},
                       capture_output=True, text=True)
    tail = (r.stdout + r.stderr).strip().splitlines()[-2:]
    print(f'   {Path(args[0]).name} {time.time() - t0:.0f}s', *tail, sep='\n     ')
    if r.returncode:
        raise RuntimeError(f'{args[0]} 실패:\n' + r.stderr[-2000:])

for v in videos:
    rel = v.relative_to(VID)
    dst = LAB / rel.parent / rel.stem              # label/train/1/
    if (dst / 'done.json').exists():
        print('skip', rel); continue
    print('==', rel, time.strftime('%H:%M:%S'))
    work = Path('/content/work') / rel.parent / rel.stem
    shutil.rmtree(work, ignore_errors=True); work.mkdir(parents=True)
    local = work / v.name
    shutil.copy(v, local)                          # 드라이브에서 바로 읽으면 느림
    env = dict(CAM_H=CAM_H.get(rel.parts[0], '1.70'), VIDEO=str(local))
    run(['pass1_detect.py', W_DET, str(local), str(work)])
    run(['pass1_lowconf.py', W_DET, str(local), str(work)])
    run(['pass1_lights.py', W_DET, W_CLS, str(local), str(work)])
    run(['pass1_lightcolor.py', str(local), str(work)])
    run(['pass1_surface.py', str(local), str(work)])
    run(['pass2_label.py', str(work), '--no-video'], env)
    local.unlink()
    dst.mkdir(parents=True, exist_ok=True)
    for f in work.iterdir():
        shutil.copy(f, dst / f.name)
    (dst / 'done.json').write_text(json.dumps(dict(video=str(rel), cam_h=env['CAM_H'], code='v5_29 + make_label_pass2',
                                                   at=time.strftime('%Y-%m-%d %H:%M:%S')), ensure_ascii=False))
    print('   saved →', dst)
print('전부 끝')

In [ ]:
# 라벨 분포 확인
import collections
for d in sorted(LAB.rglob('labels_frames.jsonl')):
    sp, mo = collections.Counter(), collections.Counter()
    for l in open(d, encoding='utf-8'):
        for o in json.loads(l)['objs']:
            sp[o['speak']] += 1; mo[o['motion']] += 1
    print(d.parent.relative_to(LAB), dict(sp), dict(mo))